# Page Replacement Algorithm Simulator
**Operating Systems Lab | Virtual Memory / Page Replacement**

**Algorithms:** FIFO, LRU, and Optimal  
**Language:** Python 3 (Jupyter Notebook)

## Objectives
- Simulate page faults and page hits for a page-reference string.
- Implement FIFO, LRU, and Optimal page replacement.
- Display each reference, frame contents, hit/fault status, and replaced page.
- Calculate hit ratio and fault ratio, then compare the algorithms.
- Investigate Belady's anomaly.

**How to use:** Run the setup/function cells first. Run the interactive input cell to enter your own data. The worked examples below verify the required manual tracing and experiments.

## 1. Algorithm logic
- **FIFO (First-In, First-Out):** replace the page that entered memory earliest.
- **LRU (Least Recently Used):** replace the page whose most recent past use is oldest.
- **Optimal:** replace the resident page whose next use is farthest in the future; if it will not be used again, it is an ideal victim. Optimal is a benchmark because a real OS cannot know future references.

All frames start empty. A **hit** occurs when the page is already in a frame. A **fault** occurs when it is absent. Ratios use the total number of page references.

In [1]:
import pandas as pd
from IPython.display import display

## 2. Implement the algorithms and output functions
The common simulator uses a different victim-selection rule for each algorithm. It returns the number of faults, hits, and a trace for every step.

In [2]:

def simulate(reference, num_frames, algorithm):
    """Return (faults, hits, trace) for FIFO, LRU, or Optimal.

    Each trace row is a dictionary containing step, page, frames, result,
    and replaced page. Frames are shown in stable slot order.
    """
    if algorithm not in {"FIFO", "LRU", "Optimal"}:
        raise ValueError("algorithm must be FIFO, LRU, or Optimal")
    if num_frames < 1:
        raise ValueError("Number of frames must be at least 1.")

    frames = []
    fifo_queue = []       # Pages ordered by when they entered memory
    last_used = {}        # Page -> index of most recent reference
    faults = 0
    hits = 0
    trace = []

    for index, page in enumerate(reference):
        replaced = "-"
        if page in frames:
            hits += 1
            result = "H"
        else:
            faults += 1
            result = "F"

            if len(frames) < num_frames:
                frames.append(page)
                fifo_queue.append(page)
            else:
                if algorithm == "FIFO":
                    victim = fifo_queue.pop(0)
                elif algorithm == "LRU":
                    # The smallest last-used index is the least recently used.
                    victim = min(frames, key=lambda p: last_used[p])
                else:
                    # Choose the page whose next use is farthest in the future.
                    future = reference[index + 1:]
                    distances = []
                    for resident_page in frames:
                        if resident_page in future:
                            distances.append(future.index(resident_page))
                        else:
                            distances.append(float("inf"))
                    # max() returns the first maximum, so ties follow frame order.
                    victim = frames[distances.index(max(distances))]

                slot = frames.index(victim)
                frames[slot] = page
                replaced = victim
                if algorithm == "FIFO":
                    fifo_queue.append(page)

        last_used[page] = index
        trace.append({
            "step": index + 1,
            "page": page,
            "frames": frames.copy(),
            "result": result,
            "replaced": replaced
        })

    return faults, hits, trace


def print_trace(reference, num_frames, algorithm):
    """Print a step-by-step table and summary for one algorithm."""
    faults, hits, trace = simulate(reference, num_frames, algorithm)
    print(f"\\n{algorithm} — {num_frames} frame(s)")
    print("-" * 72)
    print(f"{'Step':<6}{'Page':<8}{'Frame contents':<25}{'H/F':<8}{'Replaced':<10}")
    print("-" * 72)
    for row in trace:
        frame_text = "[" + ", ".join(map(str, row["frames"])) + "]"
        print(f"{row['step']:<6}{row['page']:<8}{frame_text:<25}{row['result']:<8}{row['replaced']:<10}")
    total = len(reference)
    print("-" * 72)
    print(f"Page faults: {faults}")
    print(f"Page hits:   {hits}")
    print(f"Hit ratio:   {hits / total:.2%}")
    print(f"Fault ratio: {faults / total:.2%}")
    return faults, hits, trace


def compare_algorithms(reference, num_frames, show_traces=True):
    """Run all algorithms and return a comparison DataFrame."""
    summaries = []
    all_traces = {}
    for algorithm in ("FIFO", "LRU", "Optimal"):
        if show_traces:
            faults, hits, trace = print_trace(reference, num_frames, algorithm)
        else:
            faults, hits, trace = simulate(reference, num_frames, algorithm)
        all_traces[algorithm] = trace
        total = len(reference)
        summaries.append({
            "Algorithm": algorithm,
            "Page Faults": faults,
            "Page Hits": hits,
            "Hit Ratio": f"{hits / total:.2%}",
            "Fault Ratio": f"{faults / total:.2%}"
        })
    comparison = pd.DataFrame(summaries)
    print("\\nFINAL COMPARISON")
    print(comparison.to_string(index=False))
    return comparison, all_traces


def parse_reference_string(text):
    """Parse comma- or space-separated non-negative integer page numbers."""
    parts = text.replace(",", " ").split()
    if not parts:
        raise ValueError("Enter at least one page number.")
    try:
        pages = [int(part) for part in parts]
    except ValueError:
        raise ValueError("Page references must be whole numbers separated by commas or spaces.")
    if any(page < 0 for page in pages):
        raise ValueError("Page numbers must be non-negative integers.")
    return pages




## 3. Enter your own input
This cell accepts page numbers separated by commas or spaces and validates the frame count. If an input is invalid, correct it and rerun the cell.

Example:
- Reference string: `1 2 3 1 4 2 5`
- Number of frames: `3`

### Interactive input cell (run manually)
The following code accepts your own reference string and frame count. Run it manually after the function-definition cell. It is not pre-executed in the saved notebook because it waits for keyboard input.

In [3]:
# Interactive input cell: run this cell in Jupyter to enter your own test case.
reference_text = input("Enter page reference string (comma or space separated): ")
reference = parse_reference_string(reference_text)
frames_text = input("Enter number of frames: ")
try:
    num_frames = int(frames_text)
except ValueError:
    raise ValueError("Number of frames must be a whole number.")
if num_frames < 1:
    raise ValueError("Number of frames must be at least 1.")

comparison, traces = compare_algorithms(reference, num_frames)


\nFIFO — 10 frame(s)
------------------------------------------------------------------------
Step  Page    Frame contents           H/F     Replaced  
------------------------------------------------------------------------
1     10      [10]                     F       -         
------------------------------------------------------------------------
Page faults: 1
Page hits:   0
Hit ratio:   0.00%
Fault ratio: 100.00%
\nLRU — 10 frame(s)
------------------------------------------------------------------------
Step  Page    Frame contents           H/F     Replaced  
------------------------------------------------------------------------
1     10      [10]                     F       -         
------------------------------------------------------------------------
Page faults: 1
Page hits:   0
Hit ratio:   0.00%
Fault ratio: 100.00%
\nOptimal — 10 frame(s)
------------------------------------------------------------------------
Step  Page    Frame contents           H/F     Repla

## 4. Part B — Manual tracing reference string
Required reference string: `7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2`  
Number of frames: `3`

The next cell generates the full trace for all algorithms. Compare it with the tables in the report.

In [4]:
manual_reference = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]
manual_comparison, manual_traces = compare_algorithms(manual_reference, 3)

\nFIFO — 3 frame(s)
------------------------------------------------------------------------
Step  Page    Frame contents           H/F     Replaced  
------------------------------------------------------------------------
1     7       [7]                      F       -         
2     0       [7, 0]                   F       -         
3     1       [7, 0, 1]                F       -         
4     2       [2, 0, 1]                F       7         
5     0       [2, 0, 1]                H       -         
6     3       [2, 3, 1]                F       0         
7     0       [2, 3, 0]                F       1         
8     4       [4, 3, 0]                F       2         
9     2       [4, 2, 0]                F       3         
10    3       [4, 2, 3]                F       0         
11    0       [0, 2, 3]                F       4         
12    3       [0, 2, 3]                H       -         
13    2       [0, 2, 3]                H       -         
----------------------

## 5. Part C, Experiment 1 — Belady's anomaly
Reference string: `1 2 3 4 1 2 5 1 2 3 4 5`. Compare 3 frames with 4 frames.

In [5]:
belady_reference = [1, 2, 3, 4, 1, 2, 5, 1, 2, 3, 4, 5]
belady_rows = []
for frame_count in (3, 4):
    for algorithm in ("FIFO", "LRU", "Optimal"):
        faults, hits, _ = simulate(belady_reference, frame_count, algorithm)
        belady_rows.append({"Algorithm": algorithm, "Frames": frame_count,
                            "Faults": faults, "Hits": hits})
belady_table = pd.DataFrame(belady_rows)
display(belady_table.pivot(index="Algorithm", columns="Frames", values="Faults")
        .rename(columns={3: "Faults (3 frames)", 4: "Faults (4 frames)"}))
print("FIFO increases from 9 faults (3 frames) to 10 faults (4 frames): Belady's anomaly.")

Frames,Faults (3 frames),Faults (4 frames)
Algorithm,,
FIFO,9,10
LRU,10,8
Optimal,7,6


FIFO increases from 9 faults (3 frames) to 10 faults (4 frames): Belady's anomaly.


## 6. Part C, Experiment 2 — Design your own inputs
These are two example strings, each with at least 15 references. They are evaluated with four frames.

In [6]:
lru_better_reference = [1, 1, 1, 4, 3, 5, 5, 4, 3, 2, 2, 1, 4, 2, 4, 3, 2]
fifo_equal_reference = [5, 3, 5, 2, 3, 1, 1, 2, 3, 5, 5, 2, 1, 3, 2, 3, 4, 1]

for label, sample in [("LRU better", lru_better_reference),
                      ("FIFO better/equal", fifo_equal_reference)]:
    fifo_faults = simulate(sample, 4, "FIFO")[0]
    lru_faults = simulate(sample, 4, "LRU")[0]
    print(f"{label}: {sample}")
    print(f"Frames = 4 | FIFO faults = {fifo_faults} | LRU faults = {lru_faults}\n")

LRU better: [1, 1, 1, 4, 3, 5, 5, 4, 3, 2, 2, 1, 4, 2, 4, 3, 2]
Frames = 4 | FIFO faults = 8 | LRU faults = 6

FIFO better/equal: [5, 3, 5, 2, 3, 1, 1, 2, 3, 5, 5, 2, 1, 3, 2, 3, 4, 1]
Frames = 4 | FIFO faults = 5 | LRU faults = 5



## 7. Short-answer questions
1. **Why can Optimal not be implemented in a real OS?** It needs knowledge of future page references, which the OS cannot know in advance. It is used as a theoretical benchmark.
2. **What extra bookkeeping does LRU need compared with FIFO?** LRU must track when each page was last used, using timestamps, counters, or a stack/list. FIFO mainly tracks arrival order.
3. **If frames equal the number of distinct pages, how many faults will every algorithm produce?** Assuming all those distinct pages appear in the reference string and there are enough frames for all of them, every distinct page causes one compulsory fault on its first reference. Total faults equal the number of distinct pages; later references are hits.

## 8. Conclusion
The simulator demonstrates that page replacement policy affects the number of page faults. Optimal provides a lower-bound benchmark for a given reference string and frame count. FIFO is simple but can exhibit Belady's anomaly, while LRU uses recent history and has the stack property. Results depend on the reference string, so all algorithms should be compared on identical input.